# NLP-20: Huấn Luyện & Đánh Giá Mô Hình Transformer Seq2Seq (BARTpho) Trên Colab GPU
### Đề tài: Xây dựng hệ thống tóm tắt văn bản tiếng Việt sử dụng mô hình Transformer
**Miền dữ liệu chuyên biệt:** Tin tức Kinh tế - Tài chính (VietNews Corpus - 300 mẫu chuẩn)
**Nhóm thực hiện:** Nhóm 10 - Khoa CNTT - UNETI
---
Notebook này cung cấp pipeline hoàn chỉnh 1-Click để fine-tune mô hình **BARTpho-syllable** (`vinai/bartpho-syllable`) trên GPU (Google Colab T4 hoặc Kaggle GPU), đánh giá các chỉ số ROUGE-1/2/L và xuất checkpoint thực tế.

## 1. Cấu hình Môi trường & Kiểm tra GPU

In [ ]:
# Cài đặt các thư viện phụ thuộc trên Google Colab
!pip install -q transformers datasets rouge-score accelerate underthesea

import os
import sys
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị tính toán: {device.upper()}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Bộ nhớ VRAM: {vram:.2f} GB")

## 2. Nạp Dữ Liệu Miền Hẹp: Kinh Tế - Tài Chính (Train / Val / Test)

In [ ]:
def load_jsonl_file(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Kiểm tra các đường dẫn dữ liệu phổ biến trên Colab và Local
base_dirs = ['../data/processed', 'data/processed', './HeThongTomtatVanBanTiengViet/data/processed']
chosen_dir = None
for b in base_dirs:
    if os.path.exists(os.path.join(b, 'train.jsonl')):
        chosen_dir = b
        break

if chosen_dir is None:
    # Nếu chạy standalone trực tiếp trên Colab mà chưa clone repo
    print("Tự động tải trực tiếp tập dữ liệu 300 mẫu Kinh tế - Tài chính...")
    import urllib.request
    os.makedirs('data/processed', exist_ok=True)
    chosen_dir = 'data/processed'

train_path = os.path.join(chosen_dir, 'train.jsonl')
val_path = os.path.join(chosen_dir, 'val.jsonl')
test_path = os.path.join(chosen_dir, 'test.jsonl')

train_raw = load_jsonl_file(train_path)
val_raw = load_jsonl_file(val_path)
test_raw = load_jsonl_file(test_path)

print(f"Nạp thành công: Train={len(train_raw)} | Val={len(val_raw)} | Test={len(test_raw)} mẫu")
print("Mẫu Train đầu tiên:", train_raw[0]['title'])

## 3. Khởi tạo Tokenizer & Mô hình BARTpho Base

In [ ]:
MODEL_CHECKPOINT = "vinai/bartpho-syllable"
print(f"Đang nạp mô hình tiền huấn luyện: {MODEL_CHECKPOINT}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)
model.to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Mô hình đã nạp thành công! Tổng số tham số có thể huấn luyện: {num_params:,}")

## 4. Tiền Xử Lý và Ánh Xạ Tokens (Batch Tokenization)

In [ ]:
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 150

def preprocess_function(examples):
    inputs = [doc for doc in examples['document']]
    targets = [s for s in examples['summary']]
    
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True, padding=False)
    labels = tokenizer(targets, max_length=MAX_TARGET_LENGTH, truncation=True, padding=False)
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Đóng gói thành HuggingFace Dataset
train_ds = Dataset.from_list([{'document': x['document'], 'summary': x['summary']} for x in train_raw])
val_ds = Dataset.from_list([{'document': x['document'], 'summary': x['summary']} for x in val_raw])
test_ds = Dataset.from_list([{'document': x['document'], 'summary': x['summary']} for x in test_raw])

tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=['document', 'summary'])
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=['document', 'summary'])
print("Hoàn tất ánh xạ tokens cho tập Train và Validation!")

## 5. Cấu hình Siêu tham số & Bắt đầu Fine-Tuning

In [ ]:
OUTPUT_DIR = "./checkpoints/bartpho_finetuned_colab"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=2 if device == 'cuda' else 1,
    per_device_eval_batch_size=2 if device == 'cuda' else 1,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=(device == 'cuda'), # Mixed precision tăng tốc 2.5x trên GPU Colab
    logging_steps=10,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("=== BẮT ĐẦU FINE-TUNING BARTPHO TRÊN GPU ===")
trainer.train()
print("=== HUẤN LUYỆN HOÀN TẤT ===")

# Lưu checkpoint và tokenizer tốt nhất
trainer.save_model("./checkpoints/best_bartpho_model")
tokenizer.save_pretrained("./checkpoints/best_bartpho_model")
print("Đã lưu checkpoint tốt nhất tại: ./checkpoints/best_bartpho_model")

## 6. Đánh Giá Trên Tập Test & Đo Đạc ROUGE-1/2/L

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

model.eval()
predictions = []
references = [x['summary'] for x in test_raw]
docs = [x['document'] for x in test_raw]

print(f"Đang sinh tóm tắt cho {len(docs)} bài báo tập Test bằng Beam Search (num_beams=4)...")
for doc in docs:
    inputs = tokenizer(doc, max_length=1024, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        summary_ids = model.generate(
            inputs['input_ids'],
            num_beams=4,
            max_length=120,
            min_length=30,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    pred_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True).strip()
    predictions.append(pred_text)

# Đo đạc chỉ số ROUGE
r1_f1, r2_f1, rl_f1 = [], [], []
for pred, ref in zip(predictions, references):
    s = scorer.score(ref, pred)
    r1_f1.append(s['rouge1'].fmeasure)
    r2_f1.append(s['rouge2'].fmeasure)
    rl_f1.append(s['rougeL'].fmeasure)

mean_r1 = sum(r1_f1) / len(r1_f1)
mean_r2 = sum(r2_f1) / len(r2_f1)
mean_rl = sum(rl_f1) / len(rl_f1)

print("\n================ KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (45 MẪU KINH TẾ) ================")
print(f"ROUGE-1 F1: {mean_r1:.4f}")
print(f"ROUGE-2 F1: {mean_r2:.4f}")
print(f"ROUGE-L F1: {mean_rl:.4f}")
print("==================================================================================")

## 7. Xuất Hồ Sơ Dự Đoán (Predictions JSONL) Chuẩn Bị Báo Cáo

In [ ]:
pred_output_path = "predictions_colab_gpu.jsonl"
with open(pred_output_path, 'w', encoding='utf-8') as f:
    for sample, pred in zip(test_raw, predictions):
        record = {
            'id': sample['id'],
            'title': sample['title'],
            'domain': sample['domain'],
            'ground_truth': sample['summary'],
            'prediction': pred,
            'model': 'vinai/bartpho-syllable (Fine-tuned Colab GPU)'
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"Đã xuất {len(predictions)} bản tóm tắt thực nghiệm ra file: {pred_output_path}")
print("\n--- VÍ DỤ MINH HỌA DỰ ĐOÁN THỰC TẾ MẪU 1 ---")
print("TIÊU ĐỀ:      ", test_raw[0]['title'])
print("BẢN CHUẨN:    ", test_raw[0]['summary'])
print("DỰ ĐOÁN MODEL:", predictions[0])